In [1]:
import torch
import torch.nn as nn
from einops import rearrange, reduce, repeat
from typing import Optional, Tuple
from tqdm.notebook import tqdm
import math
from typing import Iterable, Optional, Tuple, Union, List 
import torch
import transformers
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from accelerate import Accelerator
from torch.utils.data import Dataset, DataLoader
import gc
import numpy as np
EPOCHS = 5
lr = 1e-4
SEED = 42
BATCH_SIZE = 64
from einops import rearrange
import torch
import torch.nn.functional as F
from einops import pack, rearrange, unpack,reduce
from collections import OrderedDict, namedtuple
import torch.nn as nn
import math
from dataclasses import dataclass
from typing import Optional, List
import os
import numpy as np
from glob import glob
from typing import Any

In [2]:
class VisionConfig():
    def __init__(
            self,
            resolution_mode="native",
            init_method="xavier",
            num_channels=3,
            patch_size=14,
            temporal_patch_size=2,
            resize_factor = 2,
            image_size=1792,
            patch_dropout=0.0,
            attention_dropout=0.0,
            dropout=0.0,
            drop_path_rate=0.0,
            initializer_range=1e-10,
            num_hidden_layers=8,
            num_attention_heads=12,
            hidden_size=768,
            intermediate_size=3072,
            patch_embedding_bias=True,
            qk_normalization=True,
            qkv_bias=False,
            initializer_factor=0.1,
            use_pre_norm=False,
            pe_type="rope2d",
            rope_theta=10000,
            spatial_merge_size=1,
            norm_type="RMSNorm",
            hidden_act='SwiGLU',
            use_flash_attn=True,
            layer_norm_eps=1e-5,
            min_tokens=196,  #adjust it based on your gpu and data
            max_tokens=588,
            image_mean=(0.485, 0.456, 0.406),
            image_std=(0.229, 0.224, 0.225),
            relarge_ratio=1.0,
    ):
        self.resolution_mode = resolution_mode
        self.init_method = init_method
        self.pe_type = pe_type
        self.rope_theta = rope_theta
        self.temporal_patch_size = temporal_patch_size
        self.num_channels = num_channels
        self.patch_size = patch_size
        self.image_size = image_size
        self.resize_factor = resize_factor
        self.patch_dropout = patch_dropout
        self.attention_dropout = attention_dropout
        self.dropout = dropout
        self.drop_path_rate = drop_path_rate
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads 
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.patch_embedding_bias = patch_embedding_bias
        self.qk_normalization = qk_normalization
        self.qkv_bias = qkv_bias
        self.initializer_factor = initializer_factor
        self.use_pre_norm = use_pre_norm
        self.norm_type = norm_type
        self.hidden_act = hidden_act
        self.use_flash_attn = use_flash_attn
        self.layer_norm_eps = layer_norm_eps
        self.spatial_merge_size = spatial_merge_size
        self.min_tokens = min_tokens
        self.max_tokens = max_tokens
        self.image_mean = image_mean
        self.image_std = image_std
        self.relarge_ratio = relarge_ratio
        

class ImageTransform(object):
    def __init__(self, config):
        self.config = config
        self.resolution_mode = config.resolution_mode
        
        self.image_mean, self.image_std = config.image_mean, config.image_std
        self.patch_size = config.patch_size
        self.temporal_patch_size = config.temporal_patch_size
        self.spatial_merge_size = config.spatial_merge_size
        self.resize_factor = config.patch_size * config.spatial_merge_size * config.resize_factor
        self.relarge_ratio = config.relarge_ratio

        self.forced_transform = None
        self.min_pixels, self.max_pixels = None, None
        assert self.resolution_mode in ["native", "224", "378", "756"]
        if self.resolution_mode == "native":
            self.min_pixels = config.min_tokens * config.patch_size * config.patch_size
            self.max_pixels = config.max_tokens * config.patch_size * config.patch_size
        else:
            image_size = int(self.resolution_mode)
            self.forced_transform = transforms.Compose([
                    transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BICUBIC),
                    self.convert_to_rgb,
                    transforms.ToTensor(),
                    transforms.Normalize(mean=self.image_mean, std=self.image_std)
                ]
            )

    def __call__(self, images):
        
        if not isinstance(images, List):
            images = [images]  # shape of each image is [h, w, c]
        assert len(images) == 1 or len(images) % self.temporal_patch_size == 0

        if self.resolution_mode == "native":
            sample_num = 1 if len(images) == 1 else len(images) // self.temporal_patch_size
            min_pixels, max_pixels = self.min_pixels // sample_num, self.max_pixels // sample_num
            width, height = images[0].size  # (w, h)
            if self.relarge_ratio > 0 and self.relarge_ratio != 1:
                height, width = int(height * self.relarge_ratio), int(width * self.relarge_ratio)
            resized_height, resized_width = self.smart_resize(height, width, self.resize_factor, min_pixels, max_pixels)
            processed_images = []
            for image in images:
                image = self.convert_to_rgb(image)
                image = self.resize(image, size=(resized_height, resized_width), resample=Image.Resampling.BICUBIC)
                image = self.rescale(image, scale=1/255)
                image = self.normalize(image=image, mean=self.image_mean, std=self.image_std)
                processed_images.append(image)
            processed_images = np.array(processed_images)  # (num, h, w, c)
            processed_images = processed_images.transpose(0, 3, 1, 2)  # (num, c, h, w)
        else:
            processed_images = [self.forced_transform(image).numpy() for image in images]
            processed_images = np.array(processed_images)

        if processed_images.shape[0] == 1:
            processed_images = np.tile(processed_images, (self.temporal_patch_size, 1, 1, 1))

        return torch.from_numpy(processed_images) 
        
   
    def data_patchify(self,input_data):
        t, c, h, w = input_data.shape
        grid_t, grid_h, grid_w = t // self.config.temporal_patch_size, h // self.config.patch_size, w // self.config.patch_size
        grid_size = c * self.config.temporal_patch_size * self.config.patch_size * self.config.patch_size
        input_data = input_data.reshape(
            grid_t, self.config.temporal_patch_size, c, 
            grid_h // self.config.spatial_merge_size, self.config.spatial_merge_size, self.config.patch_size, 
            grid_w // self.config.spatial_merge_size, self.config.spatial_merge_size, self.config.patch_size
        )
        input_data = input_data.permute(0, 3, 6, 4, 7, 2, 1, 5, 8)
        input_data = input_data.reshape(grid_t * grid_h * grid_w, grid_size).contiguous()
        grid_shape = (grid_t, grid_h, grid_w)
        return input_data, grid_shape
    
    @staticmethod
    def convert_to_rgb(image):
        if not isinstance(image, Image.Image):
            return image
        # `image.convert("RGB")` would only work for .jpg images, as it creates a wrong background
        # for transparent images. The call to `alpha_composite` handles this case
        if image.mode == "RGB":
            return image
        image_rgba = image.convert("RGBA")
        background = Image.new("RGBA", image_rgba.size, (255, 255, 255))
        alpha_composite = Image.alpha_composite(background, image_rgba)
        alpha_composite = alpha_composite.convert("RGB")
        return alpha_composite
    
    @staticmethod
    def resize(image, size, resample, return_numpy: bool = True) -> np.ndarray:
        """
        Resizes `image` to `(height, width)` specified by `size` using the PIL library.
        """
        if not len(size) == 2:
            raise ValueError("size must have 2 elements")
        assert isinstance(image, Image.Image)
        height, width = size
        resample = resample if resample is not None else Image.Resampling.BILINEAR
        # PIL images are in the format (width, height)
        resized_image = image.resize((width, height), resample=resample, reducing_gap=None)
        if return_numpy:
            resized_image = np.array(resized_image)
            resized_image = np.expand_dims(resized_image, axis=-1) if resized_image.ndim == 2 else resized_image
        return resized_image

    @staticmethod
    def rescale(image: np.ndarray, scale: float, dtype: np.dtype = np.float32) -> np.ndarray:
        if not isinstance(image, np.ndarray):
            raise TypeError(f"Input image must be of type np.ndarray, got {type(image)}")
        rescaled_image = image * scale
        rescaled_image = rescaled_image.astype(dtype)
        return rescaled_image

    @staticmethod
    def normalize(image, mean, std) -> np.ndarray:
        """
        Normalizes `image` using the mean and standard deviation specified by `mean` and `std`.
        image = (image - mean) / std
        """
        if not isinstance(image, np.ndarray):
            raise ValueError("image must be a numpy array")
        num_channels = image.shape[-1]
        # We cast to float32 to avoid errors that can occur when subtracting uint8 values.
        # We preserve the original dtype if it is a float type to prevent upcasting float16.
        if not np.issubdtype(image.dtype, np.floating):
            image = image.astype(np.float32)
        if isinstance(mean, Iterable):
            if len(mean) != num_channels:
                raise ValueError(f"mean must have {num_channels} elements if it is an iterable, got {len(mean)}")
        else:
            mean = [mean] * num_channels
        mean = np.array(mean, dtype=image.dtype)
        if isinstance(std, Iterable):
            if len(std) != num_channels:
                raise ValueError(f"std must have {num_channels} elements if it is an iterable, got {len(std)}")
        else:
            std = [std] * num_channels
        std = np.array(std, dtype=image.dtype)
        image = (image - mean) / std
        return image
    
    @staticmethod
    def smart_resize(height, width, factor, min_pixels, max_pixels):
        """ 
        1. Both dimensions (height and width) are divisible by 'factor'.
        2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].
        3. The aspect ratio of the image is maintained as closely as possible.
        """
        if height < factor or width < factor:
            if height < factor:
                ratio = factor / height
                height, width = factor, int(ratio * width) + 1
            if width < factor:
                ratio = factor / width
                width, height = factor, int(ratio * height) + 1
        h_bar = round(height / factor) * factor #make multiple of factor
        w_bar = round(width / factor) * factor
        if h_bar * w_bar > max_pixels:
            #we need to shrink the image
            beta = math.sqrt((height * width) / max_pixels)
            h_bar = math.floor(height / beta / factor) * factor
            w_bar = math.floor(width / beta / factor) * factor
        elif h_bar * w_bar < min_pixels:
             #we need to  expand the image
            beta = math.sqrt(min_pixels / (height * width))
            h_bar = math.ceil(height * beta / factor) * factor
            w_bar = math.ceil(width * beta / factor) * factor
        return h_bar, w_bar


import torch
import triton
import triton.language as tl
import math

@triton.jit
def _attn_fwd_varlen_kernel(
    Q, K, V, cu_seqlens, sm_scale, L, Out,
    stride_qt, stride_qh, stride_qk,
    stride_kt, stride_kh, stride_kk,
    stride_vt, stride_vh, stride_vk,
    stride_ot, stride_oh, stride_ok,
    HEADS, 
    HEAD_DIM: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
    BLOCK_DMODEL: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_bh = tl.program_id(1)
    batch_idx = pid_bh // HEADS
    head_idx = pid_bh % HEADS

    # bounds
    start_idx = tl.load(cu_seqlens + batch_idx)
    end_idx = tl.load(cu_seqlens + batch_idx + 1)
    seq_len = end_idx - start_idx

    if pid_m * BLOCK_M >= seq_len:
        return

    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rk = tl.arange(0, BLOCK_DMODEL)
    rn_ =  tl.arange(0, BLOCK_N)
    
    # Q Pointer ->  (Base + TokenOffset + HeadOffset + DimOffset) same pattern for K, V
    q_ptr = Q + (start_idx + rm[:, None]) * stride_qt + head_idx * stride_qh + rk[None, :] * stride_qk
    q_mask = (rm[:, None] < seq_len) & (rk[None, :] < HEAD_DIM)
    q = tl.load(q_ptr, mask=q_mask, other=0.0) # load [N,D]

    m_i = tl.full([BLOCK_M], -float("inf"), tl.float32)
    l_i = tl.zeros([BLOCK_M], tl.float32)
    acc = tl.zeros([BLOCK_M, BLOCK_DMODEL], tl.float32)
    

    for start_n in range(0, seq_len, BLOCK_N):
        rn = start_n + rn_
        
        # K load as [D, N])
        k_ptr = K + (start_idx + rn[None, :]) * stride_kt + head_idx * stride_kh + rk[:, None] * stride_kk
        k_mask = (rn[None, :] < seq_len) & (rk[:, None] < HEAD_DIM)
        k = tl.load(k_ptr, mask=k_mask, other=0.0)

        # V Load ([N, D])
        v_ptr = V + (start_idx + rn[:, None]) * stride_vt + head_idx * stride_vh + rk[None, :] * stride_vk
        v_mask = (rn[:, None] < seq_len) & (rk[None, :] < HEAD_DIM)
        v = tl.load(v_ptr, mask=v_mask, other=0.0)

        # Attention 
        qk = tl.dot(q, k,out_dtype=tl.float32) * sm_scale #[N,N]
        qk = tl.where(rn[None, :] < seq_len, qk, -float("inf"))

        m_ij = tl.maximum(m_i, tl.max(qk, axis=1))
        p = tl.exp(qk - m_ij[:, None])
        l_ij = tl.sum(p, axis=1)

        alpha = tl.exp(m_i - m_ij)
        acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v,out_dtype=tl.float32) #[N,N]dot[N,D]> [N,D]
        l_i = l_i * alpha + l_ij
        m_i = m_ij

    # Store Output
    off_o = (start_idx + rm[:, None]) * stride_ot + head_idx * stride_oh + rk[None, :] * stride_ok
    tl.store(Out + off_o, (acc / l_i[:, None]).to(Out.dtype.element_ty), mask=q_mask)
    
    # Store (LogSumExp) for backward
    off_l = (start_idx + rm) * HEADS + head_idx
    tl.store(L + off_l, m_i + tl.log(l_i), mask=rm < seq_len)

@triton.jit
def _bwd_preprocess_varlen(
    Out, dOut, D_out, T_size, HEADS, stride_ot, stride_oh, stride_ok, 
    BLOCK_M: tl.constexpr, HEAD_DIM: tl.constexpr, BLOCK_DMODEL: tl.constexpr
):
    pid_m = tl.program_id(0)
    pid_bh = tl.program_id(1)
    head_idx = pid_bh % HEADS
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rk = tl.arange(0, BLOCK_DMODEL)
    
    mask_m = rm < T_size
    off = rm[:, None] * stride_ot + head_idx * stride_oh + rk[None, :] * stride_ok
    mask_md = mask_m[:, None] & (rk[None, :] < HEAD_DIM)
    
    o = tl.load(Out + off, mask=mask_md, other=0.0)
    do = tl.load(dOut + off, mask=mask_md, other=0.0)
    
    delta = tl.sum(o * do, axis=1)
    # Store delta as (T, H)
    tl.store(D_out + rm * HEADS + head_idx, delta, mask=mask_m)


@triton.jit
def _attn_bwd_dq_varlen_kernel(
    Q, K, V, cu_seqlens, sm_scale, dO, dQ, L, D,
    stride_qt, stride_qh, stride_qk, stride_kt, stride_kh, stride_kk,
    stride_vt, stride_vh, stride_vk, HEADS, 
    HEAD_DIM: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_DMODEL: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_bh = tl.program_id(1)
    batch_idx = pid_bh // HEADS
    head_idx = pid_bh % HEADS
    start_idx = tl.load(cu_seqlens + batch_idx)
    end_idx = tl.load(cu_seqlens + batch_idx + 1)
    seq_len = end_idx - start_idx

    if pid_m * BLOCK_M >= seq_len: return
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rk = tl.arange(0, BLOCK_DMODEL)
    mask_md = (rm[:, None] < seq_len) & (rk[None, :] < HEAD_DIM)
    rn_ = tl.arange(0, BLOCK_N)
    
    q = tl.load(Q + (start_idx + rm[:, None]) * stride_qt + head_idx * stride_qh + rk[None, :] * stride_qk, mask=mask_md, other=0.0)
    do = tl.load(dO + (start_idx + rm[:, None]) * stride_qt + head_idx * stride_qh + rk[None, :] * stride_qk, mask=mask_md, other=0.0)
    lse = tl.load(L + (start_idx + rm) * HEADS + head_idx, mask=rm < seq_len)
    di = tl.load(D + (start_idx + rm) * HEADS + head_idx, mask=rm < seq_len)
    
    dq = tl.zeros([BLOCK_M, BLOCK_DMODEL], tl.float32)
  
    for start_n in range(0, seq_len, BLOCK_N):
        rn = start_n + rn_
        k = tl.load(K + (start_idx + rn[None, :]) * stride_kt + head_idx * stride_kh + rk[:, None] * stride_kk, 
                    mask=(rn[None, :] < seq_len) & (rk[:, None] < HEAD_DIM), other=0.0)
        v = tl.load(V + (start_idx + rn[:, None]) * stride_vt + head_idx * stride_vh + rk[None, :] * stride_vk, 
                    mask=(rn[:, None] < seq_len) & (rk[None, :] < HEAD_DIM), other=0.0)
        
        qk = tl.dot(q, k,out_dtype=tl.float32) * sm_scale
        p = tl.exp(qk - lse[:, None])
        p = tl.where(rn[None, :] < seq_len, p, 0.0)
        
        dp = (tl.dot(do, tl.trans(v),out_dtype=tl.float32) - di[:, None]) * p
        dq += tl.dot(dp.to(k.dtype), tl.trans(k),out_dtype=tl.float32)
        
    tl.store(dQ + (start_idx + rm[:, None]) * stride_qt + head_idx * stride_qh + rk[None, :] * stride_qk, (dq * sm_scale).to(dQ.dtype.element_ty), mask=mask_md)


@triton.jit
def _attn_bwd_dkv_varlen_kernel(
    Q, K, V, cu_seqlens, sm_scale, dO, dK, dV, L, D,
    stride_qt, stride_qh, stride_qk, stride_kt, stride_kh, stride_kk,
    stride_vt, stride_vh, stride_vk, HEADS,
    HEAD_DIM: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_DMODEL: tl.constexpr,
):
    pid_n = tl.program_id(0)
    pid_bh = tl.program_id(1)
    batch_idx = pid_bh // HEADS
    head_idx = pid_bh % HEADS
    start_idx = tl.load(cu_seqlens + batch_idx)
    end_idx = tl.load(cu_seqlens + batch_idx + 1)
    seq_len = end_idx - start_idx

    if pid_n * BLOCK_N >= seq_len: return
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    rk = tl.arange(0, BLOCK_DMODEL)
    mask_nd = (rn[:, None] < seq_len) & (rk[None, :] < HEAD_DIM)
    rm_ = tl.arange(0, BLOCK_M)
    
    k = tl.load(K + (start_idx + rn[:, None]) * stride_kt + head_idx * stride_kh + rk[None, :] * stride_kk, mask=mask_nd, other=0.0)
    v = tl.load(V + (start_idx + rn[:, None]) * stride_vt + head_idx * stride_vh + rk[None, :] * stride_vk, mask=mask_nd, other=0.0)
    
    dk = tl.zeros([BLOCK_N, BLOCK_DMODEL], tl.float32)
    dv = tl.zeros([BLOCK_N, BLOCK_DMODEL], tl.float32)
    
    
    for start_m in range(0, seq_len, BLOCK_M):
        rm = start_m + rm_
        q = tl.load(Q + (start_idx + rm[:, None]) * stride_qt + head_idx * stride_qh + rk[None, :] * stride_qk, 
                    mask=(rm[:, None] < seq_len) & (rk[None, :] < HEAD_DIM), other=0.0)
        do = tl.load(dO + (start_idx + rm[:, None]) * stride_qt + head_idx * stride_qh + rk[None, :] * stride_qk, 
                     mask=(rm[:, None] < seq_len) & (rk[None, :] < HEAD_DIM), other=0.0)
        lse = tl.load(L + (start_idx + rm) * HEADS + head_idx, mask=rm < seq_len)
        di = tl.load(D + (start_idx + rm) * HEADS + head_idx, mask=rm < seq_len)
        
        qk = tl.dot(q, tl.trans(k),out_dtype=tl.float32) * sm_scale
        
        p = tl.exp(qk - lse[:, None])
        p = tl.where(rm[:, None] < seq_len, p, 0.0)
        
        dv += tl.dot(tl.trans(p.to(do.dtype)), do,out_dtype=tl.float32)
        dp = (tl.dot(do, tl.trans(v),out_dtype=tl.float32) - di[:, None]) * p
        dk += tl.dot(tl.trans(dp.to(q.dtype)), q,out_dtype=tl.float32)
        
    tl.store(dK + (start_idx + rn[:, None]) * stride_kt + head_idx * stride_kh + rk[None, :] * stride_kk, (dk * sm_scale).to(dK.dtype.element_ty), mask=mask_nd)
    tl.store(dV + (start_idx + rn[:, None]) * stride_vt + head_idx * stride_vh + rk[None, :] * stride_vk, dv.to(dV.dtype.element_ty), mask=mask_nd)

class FlashVarLen(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, cu_seqlens, sm_scale):
        T, H, D = q.shape
        num_seqs = len(cu_seqlens) - 1
        # max_s 
        max_s = (cu_seqlens[1:] - cu_seqlens[:-1]).max().item()
        
        BLOCK_M, BLOCK_N = 32, 32
        out = torch.empty_like(q)
        L = torch.empty((T, H), device=q.device, dtype=torch.float32)
        
        grid = (triton.cdiv(max_s, BLOCK_M), num_seqs * H)
        _attn_fwd_varlen_kernel[grid](
            q, k, v, cu_seqlens, sm_scale, L, out,
            q.stride(0), q.stride(1), q.stride(2),
            k.stride(0), k.stride(1), k.stride(2),
            v.stride(0), v.stride(1), v.stride(2),
            out.stride(0), out.stride(1), out.stride(2),
            H, HEAD_DIM=D, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, 
            BLOCK_DMODEL=min(64, triton.next_power_of_2(D))
        )
        ctx.save_for_backward(q, k, v, cu_seqlens, L, out)
        ctx.sm_scale = sm_scale
        return out

    @staticmethod
    def backward(ctx, do):
        q, k, v, cu_seqlens, L, out = ctx.saved_tensors
        T, H, D = q.shape
        num_seqs = len(cu_seqlens) - 1
         # max_s 
        max_s = (cu_seqlens[1:] - cu_seqlens[:-1]).max().item()
        
        dq, dk, dv = torch.zeros_like(q), torch.zeros_like(k), torch.zeros_like(v)
        delta = torch.empty((T, H), device=q.device, dtype=torch.float32)
        
        BM, BN = 32, 32
        BLOCK_DMODEL = min(64, triton.next_power_of_2(D))
        
        # Delta
        _bwd_preprocess_varlen[(triton.cdiv(T, BM), H)](
            out, do, delta, T, H, 
            out.stride(0), out.stride(1), out.stride(2), 
            BLOCK_M=BM, HEAD_DIM=D, BLOCK_DMODEL=BLOCK_DMODEL
        )
        
        # dQ
        grid_dq = (triton.cdiv(max_s, BM), num_seqs * H)
        _attn_bwd_dq_varlen_kernel[grid_dq](
            q, k, v, cu_seqlens, ctx.sm_scale, do, dq, L, delta,
            q.stride(0), q.stride(1), q.stride(2),
            k.stride(0), k.stride(1), k.stride(2),
            v.stride(0), v.stride(1), v.stride(2),
            H, HEAD_DIM=D, BLOCK_M=BM, BLOCK_N=BN, BLOCK_DMODEL=BLOCK_DMODEL
        )
        
        # dK, dV
        grid_dkv = (triton.cdiv(max_s, BN), num_seqs * H)
        _attn_bwd_dkv_varlen_kernel[grid_dkv](
            q, k, v, cu_seqlens, ctx.sm_scale, do, dk, dv, L, delta,
            q.stride(0), q.stride(1), q.stride(2),
            k.stride(0), k.stride(1), k.stride(2),
            v.stride(0), v.stride(1), v.stride(2),
            H, HEAD_DIM=D, BLOCK_M=BM, BLOCK_N=BN, BLOCK_DMODEL=BLOCK_DMODEL
        )
        return dq, dk, dv, None, None

class TritonVerlenAttention(torch.nn.Module):
    def __init__(self, sm_scale=None):
        super().__init__()
        self.sm_scale = sm_scale

    def forward(self, q, k, v,cu_seqlens):
        """
        Args:
            q, k, v: Tensors of shape (Batch, Heads, Seq_Len, Head_Dim)
            cu_seqlens: Tensor of shape (no of seqs+1)
        """
        scale = self.sm_scale if self.sm_scale is not None else q.size(-1)**-0.5
        
        return FlashVarLen.apply(q, k, v, cu_seqlens, scale)
        
def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb_vision(tensor: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
    orig_dtype = tensor.dtype
    tensor = tensor.float()
    cos = freqs.cos()
    sin = freqs.sin()
    cos = cos.unsqueeze(1).repeat(1, 1, 2).unsqueeze(0).float()
    sin = sin.unsqueeze(1).repeat(1, 1, 2).unsqueeze(0).float()
    output = (tensor * cos) + (rotate_half(tensor) * sin)
    output = output.to(orig_dtype)
    return output


class VisionRotaryEmbedding2D(nn.Module):
    def __init__(self, dim: int, theta: float = 10000.0) -> None:
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward_(self, seqlen: int) -> torch.Tensor:
        seq = torch.arange(seqlen, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(seq, self.inv_freq)
        return freqs
    
    def forward(self, grid_shapes, spatial_merge_size=2):
        pos_ids = []
        s = spatial_merge_size
        for t, h, w in grid_shapes:
            hpos_ids = torch.arange(h).unsqueeze(1).expand(-1, w)
            hpos_ids = hpos_ids.reshape(h // s, s, w // s, s)
            hpos_ids = hpos_ids.permute(0, 2, 1, 3)
            hpos_ids = hpos_ids.flatten()
            # [0,0,1,1, 0,0,1,1, 2,2,3,3, 2,2,3,3] it is now ordered block-wise.
            wpos_ids = torch.arange(w).unsqueeze(0).expand(h, -1)
            wpos_ids = wpos_ids.reshape(h // s, s, w // s, s)
            wpos_ids = wpos_ids.permute(0, 2, 1, 3)
            wpos_ids = wpos_ids.flatten()
            pos_ids.append(torch.stack([hpos_ids, wpos_ids], dim=-1).repeat(t, 1))
        pos_ids = torch.cat(pos_ids, dim=0)
        max_grid_size = torch.tensor(grid_shapes).max()
        rotary_pos_emb_full = self.forward_(max_grid_size)
        rotary_pos_emb = rotary_pos_emb_full[pos_ids].flatten(1)
        return rotary_pos_emb


class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.inner_hidden_size = int(config.intermediate_size * 2 / 3)
        self.act = nn.SiLU()
        self.fc1 = nn.Linear(config.hidden_size, self.inner_hidden_size)
        self.fc2 = nn.Linear(self.inner_hidden_size, config.hidden_size)
        self.fc3 = nn.Linear(config.hidden_size, self.inner_hidden_size)
        self.norm = nn.RMSNorm(self.inner_hidden_size, eps=config.layer_norm_eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden_states = self.fc1(x)
        hidden_states = self.act(hidden_states)
        hidden_states = self.fc2(self.norm(hidden_states * self.fc3(x)))
        return hidden_states

class VisionEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.patch_size = config.patch_size
        self.temporal_patch_size = config.temporal_patch_size
        self.kernel_size = [self.temporal_patch_size, self.patch_size, self.patch_size]
        self.use_bias = config.patch_embedding_bias
        self.patch_embedding = nn.Conv3d(
            in_channels=3, out_channels=self.embed_dim, kernel_size=self.kernel_size, stride=self.kernel_size, bias=self.use_bias)

    def forward(self, pixel_values: torch.FloatTensor, **kwargs) -> torch.Tensor:  
        pixel_values = pixel_values.view(-1, 3, *self.kernel_size)
        patch_embeds = self.patch_embedding(pixel_values)
        embeddings = patch_embeds.view(-1, self.embed_dim)
        self.num_patches = embeddings.shape[1]
        return embeddings


class Attention(nn.Module):
    """Multi-headed attention from 'Attention Is All You Need' paper"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.embed_dim // self.num_heads
        assert config.use_flash_attn is True,  "FlashAttention must be used!"
        assert self.head_dim * self.num_heads == self.embed_dim
        self.attn = TritonVerlenAttention(self.embed_dim**-0.5)
        
        self.qkv = nn.Linear(self.embed_dim, 3 * self.embed_dim, bias=config.qkv_bias)
        self.proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.proj_drop = nn.Dropout(config.dropout)
        


    def forward(self, hidden_states: torch.Tensor, **kwargs) -> torch.Tensor: 
        key_padding_mask = kwargs.get("key_padding_mask", None)
        rotary_pos_emb = kwargs["rotary_pos_emb"]

        qkv = self.qkv(hidden_states)
        qkv = rearrange(qkv, '... (three h d) -> ... three h d', three=3, h=self.num_heads)
        bind_dim = qkv.dim() - 3
        
        q, k, v = qkv.unbind(bind_dim)
        q = apply_rotary_pos_emb_vision(q.unsqueeze(0), rotary_pos_emb).squeeze(0)
        k = apply_rotary_pos_emb_vision(k.unsqueeze(0), rotary_pos_emb).squeeze(0)
        
        cu_seqlens = kwargs['cu_seqlens']
            
        cu_seqlens = cu_seqlens.to(torch.int32).contiguous()
        q = q.float()
        k = k.float()
        v = v.float()
        out =  self.attn(q.contiguous(),k.contiguous(),v.contiguous(),cu_seqlens)
        out = out.to(hidden_states.dtype)
        out = rearrange(out, '... h d -> ... (h d)')
        out = self.proj(out)
        out = self.proj_drop(out)
        return out
        
class VisionEncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed_dim = config.hidden_size
        assert config.hidden_act == "SwiGLU"

        self.attn = Attention(config)
        self.norm1 = nn.RMSNorm(self.embed_dim, eps=config.layer_norm_eps)
        self.norm2 = nn.RMSNorm(self.embed_dim, eps=config.layer_norm_eps)
        self.mlp = SwiGLU(config)

    def forward(self, hidden_states: torch.Tensor, **kwargs):
        hidden_states = hidden_states + self.attn(self.norm1(hidden_states), **kwargs )
        hidden_states = hidden_states + self.mlp(self.norm2(hidden_states))
        return hidden_states

class VisionEncoder(nn.Module):
    """ Transformer encoder consisting of `config.num_hidden_layers` self attention layers. """
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.layers = nn.ModuleList([VisionEncoderLayer(config) for idx in range(config.num_hidden_layers)])
        head_dim = config.hidden_size // config.num_attention_heads
        self.rotary_pos_emb = VisionRotaryEmbedding2D(head_dim // 2, theta=self.config.rope_theta)
       

    def forward(self, inputs_embeds, output_hidden_states = False, **kwargs):
        kwargs["rotary_pos_emb"] = self.rotary_pos_emb(kwargs["grid_shapes"], self.config.spatial_merge_size)
        
        hidden_states = inputs_embeds
        for idx, encoder_layer in enumerate(self.layers):
            hidden_states = encoder_layer(hidden_states, **kwargs)

        return hidden_states

def mean_pool_varlen(hidden_states, cu_seqlens):

    device = hidden_states.device
    B = len(cu_seqlens) - 1
    D = hidden_states.size(-1)

    lengths = cu_seqlens[1:] - cu_seqlens[:-1]

    seq_ids = torch.repeat_interleave(
        torch.arange(B, device=device),
        lengths
    )

    pooled = torch.zeros(
        B,
        D,
        device=device,
        dtype=hidden_states.dtype
    )

    pooled.index_add_(0, seq_ids, hidden_states)

    pooled = pooled / lengths.unsqueeze(-1)

    return pooled
    
class VisionModel(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__()
        config = VisionConfig()
        self.config = config

        self.embeddings = VisionEmbeddings(config)
        self.encoder = VisionEncoder(config)
        
    def forward(self, pixel_values, **kwargs):
        assert len(pixel_values.shape) == 2, "(batch_num_tokens, hidden_size)"
        # assert "grid_shapes" in kwargs, "grid_shapes: [(t, h, w), ..., (t, h, w)]"
        hidden_states = self.embeddings(pixel_values, **kwargs)
        last_hidden_state = self.encoder(hidden_states, **kwargs)
        pooled = mean_pool_varlen(last_hidden_state.squeeze(0),kwargs["cu_seqlens"])
        return pooled

In [3]:
from torch.utils.data import Dataset, random_split, DataLoader
from PIL import Image
import torchvision.transforms as T

class myDataset(Dataset):
    def __init__(self, root_dir,df, transform=None):
        self.transform = transform
        self.root_dir = root_dir  
        self.image_path = df['image_name'].values
        self.label = df['label'].values
        self.image_transform = ImageTransform(VisionConfig())
      

    def __len__(self):
        return len(self.label)       

    def __getitem__(self, idx):
        img_path = self.image_path[idx]
        img = Image.open(self.root_dir+'/'+img_path).convert("RGB")
       
        # if self.transform:
        #     img = self.transform(img) 
            
        data_item = self.image_transform(img)
        img, grid_shape = self.image_transform.data_patchify(data_item)
            
        
        return img,grid_shape, self.label[idx]  
        
def collate_fn(batch):
    data_inputs,grid_shapes, labels = zip(*batch)
    seqlens = torch.tensor([item.shape[0] for item in data_inputs], dtype=torch.long)
    data_inputs = torch.concatenate(data_inputs, dim=0)
    
    cu_seqlens = torch.zeros(len(seqlens) + 1, dtype=torch.long)
    cu_seqlens[1:] = torch.cumsum(seqlens, dim=0)
    # take data_inputs and grid_shape  from batch
    labels = torch.tensor(labels, dtype=torch.long)
    

    return {
    
        "pixel_values": data_inputs,
        "cu_seqlens":cu_seqlens,
        "grid_shapes": grid_shapes,
        "label": labels
    }


imagenet_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

In [4]:
import pandas as pd
df = pd.read_csv('../input/scene-classification/train-scene classification/train.csv')

In [5]:
df.head()

,image_name,label
0,0.jpg,0
1,1.jpg,4
2,2.jpg,5
3,4.jpg,0
4,7.jpg,4


In [6]:
df.shape

(17034, 2)

In [7]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2)

In [8]:
imagenet_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # mean and std values of the Imagenet Dataset so that pretrained models could also be used

#setting a set of transformations to transform the images 
transform= T.Compose([
                      T.RandomHorizontalFlip(),
                      T.RandomRotation(2),
                      T.ToTensor(),
                      T.Normalize(*imagenet_stats)])
test_transform = T.Compose([
                      T.ToTensor(),
                      T.Normalize(*imagenet_stats)])

In [9]:
data_dir = '../input/scene-classification/train-scene classification/train'
train_loader = DataLoader(myDataset(data_dir,train, transform = transform),collate_fn=collate_fn,batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader =  DataLoader(myDataset(data_dir,test, transform = test_transform),collate_fn=collate_fn,batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [10]:
 df['label'].nunique()

6

In [11]:
class VitModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = VisionModel()
        self.output = nn.Linear(768,6)

    def forward(self, pixel_values, **kwargs):
        sequence_output = self.model(pixel_values, **kwargs)
        logits = self.output(sequence_output)
        return logits

In [12]:
def valid_func(model,val_loader,val_bar):
    model.eval()
    loss_fn = torch.nn.CrossEntropyLoss()
    PROB = []
    TARGETS = []
    losses = []
    PREDS = []
   
    for batch_idx,data in enumerate(val_loader):
        val_bar.update(1)
        pixel_values = data["pixel_values"].cuda()
        cu_seqlens = data["cu_seqlens"].cuda()
        grid_shapes = data["grid_shapes"]
        targets = data["label"].long().view(-1).cuda()
        with torch.no_grad():
            pred = model( pixel_values=pixel_values, cu_seqlens=cu_seqlens, grid_shapes=grid_shapes)

        PREDS += [torch.argmax(pred, 1).detach().cpu()]
        TARGETS += [targets.detach().cpu()]

        loss = loss_fn (pred, targets)
        losses.append(loss.item())
        val_bar.set_description(f'step: {batch_idx+1} loss: {"%.4f" % loss}')

    PREDS = torch.cat(PREDS).cpu().numpy()
    TARGETS = torch.cat(TARGETS).cpu().numpy()
    accuracy = (PREDS==TARGETS).mean()
   
    loss_valid = np.mean(losses)
    return loss_valid, accuracy

In [13]:

from transformers import get_linear_schedule_with_warmup
def single_gpu(name):
    use_amp = True
    debug = False
    gc.collect()
    best_epoch_loss = np.inf

    net = VitModel()
    net.cuda()
   
    accelerator = Accelerator(log_with="tensorboard", project_dir=".")
    Config = {
    "num_epoch": EPOCHS,
    "learning_rate": lr,
    "loss_function": str(torch.nn.CrossEntropyLoss)}

    accelerator.init_trackers(f"{name}_project", config=Config)
    accumulation_steps = 2
    loss_fn = torch.nn.CrossEntropyLoss()
    no_decay = ['bias', 'layernorm.weight','layernorm.bias']
    optimizer_grouped_parameters = [
    {'params': [p for n, p in net.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in net.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=lr)
   
    num_train_optimization_steps = int(EPOCHS * len(train_loader) / accumulation_steps)

    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=1, factor= 0.8)
   
    epoch_check = len(train_loader)
    total_step = epoch_check*EPOCHS
    train_bar = tqdm(total=total_step, dynamic_ncols=True)
    val_bar = tqdm(total=len(val_loader),leave = True, dynamic_ncols=True)
    t_step = 1
    for epoch in range(EPOCHS):
        avg_loss = 0.0
        net.train()
        loss_list = []
        for step, data in enumerate(train_loader):
            train_bar.update(1)
            optimizer.zero_grad()
            pixel_values = data["pixel_values"].cuda()
            cu_seqlens = data["cu_seqlens"].cuda()
            grid_shapes = data["grid_shapes"]
            targets = data["label"].long().view(-1).cuda()

            pred = net(pixel_values=pixel_values, cu_seqlens=cu_seqlens, grid_shapes=grid_shapes)
            loss = loss_fn(pred, targets)
            loss.backward()
            optimizer.step()
            
            
            accelerator.log({"training_loss_step": loss}, step= t_step)
            t_step+=1


            loss_list.append(loss.detach().cpu().item())
        avg_loss = np.round(np.mean(loss_list), 4)
        accelerator.log({"train_epoch":  avg_loss}, step= epoch)
        print(f'train_epoch-{epoch} loss: {"%.4f" % avg_loss}')
        vloss,vaccuracy = valid_func(net,val_loader,val_bar)
        accelerator.log({"vloss_epoch": loss}, step= epoch)
        accelerator.log({"vaccuracy_epoch": vaccuracy}, step= epoch)
        print(f'Epoc: {epoch} loss: {"%.4f" % vloss},accuracy: {"%.4f" % vaccuracy}')
        val_bar.reset()
        lr_scheduler.step(vloss)
    accelerator.end_training()

In [14]:
 single_gpu(name = 'vit_classification')

2026-05-16 11:59:07.362983: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778932747.803945      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778932747.920595      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778932749.013008      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778932749.013072      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778932749.013076      23 computation_placer.cc:177] computation placer alr

  0%|          | 0/1065 [00:00<?, ?it/s]

  0%|          | 0/54 [00:00<?, ?it/s]

train_epoch-0 loss: 1.1196
Epoc: 0 loss: 0.7263,accuracy: 0.7203
train_epoch-1 loss: 0.6734
Epoc: 1 loss: 0.5955,accuracy: 0.7778
train_epoch-2 loss: 0.5267
Epoc: 2 loss: 0.5064,accuracy: 0.8101
train_epoch-3 loss: 0.4280
Epoc: 3 loss: 0.5567,accuracy: 0.8060
train_epoch-4 loss: 0.3354
Epoc: 4 loss: 0.5729,accuracy: 0.8054
